# Knowledge Graphs and Semantic Technologies -- RDFS tutorial

Here we'll learn the basics of RDFS (RDF Schema) and how to perform basic RDFS reasoning with rdflib (documentation [here](https://rdflib.readthedocs.io/en/stable/)) and owlrl (documentation [here](https://owl-rl.readthedocs.io/en/latest/)).

## Imports

owlrl is a library implementing basic RDFS and OWL reasoning on top of rdflib. We'll install and import its relevant symbols.

In [1]:
import sys
!{sys.executable} -m pip install rdflib owlrl

from rdflib import Graph, RDFS, RDF, URIRef, Namespace, Literal
from owlrl import DeductiveClosure, RDFS_Semantics
from rdflib.namespace import OWL

## Loading RDFS graphs

Your file `yourRDF.ttl` already contains a basic Knowledge Graph in RDF with some RDFS semantics

First, we are going to add some RDFS semantics, and inspect the graph as-is; this is also called the "asserted graph"

**Exercise 1** 
1. add additional triples using the RDFS semantics: have a look [here](https://www.w3.org/TR/rdf-schema/), and use domain and range, subPropertyOf, and Class, to say more about the instances in your graph
2. load yourRDF graph
3. print the classes in your graph
4. print the properties of a specific class in yourRDF graph
5. print all instances in yourRDF graph (all objects that have a type) 
6. explain what constitutes a vocabulary in RDF

In [2]:
#define graph
g = Graph()

#define example namespace
EX = Namespace("http://example.org/restaurant#")
g.bind("ex", EX)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)


#2
g.parse("./data/ingredients.ttl", format="turtle") 

#1
# Add triples using store's add method.

#classes 
g.add((EX.Restaurant, RDF.type, RDFS.Class))
g.add((EX.Chef, RDF.type, RDFS.Class))
g.add((EX.Dish, RDF.type, RDFS.Class))
g.add((EX.Ingredient, RDF.type, RDFS.Class))

#properties
g.add((EX.hasChef, RDF.type, RDF.Property))
g.add((EX.makesDish, RDF.type, RDF.Property))
g.add((EX.hasIngredient, RDF.type, RDF.Property))
g.add((EX.usesIngredient, RDF.type, RDF.Property))

#domain & range
g.add((EX.hasChef, RDFS.domain, EX.Restaurant))
g.add((EX.hasChef, RDFS.range, EX.Chef))

g.add((EX.makesDish, RDFS.domain, EX.Chef))
g.add((EX.makesDish, RDFS.range, EX.Dish))

g.add((EX.hasIngredient, RDFS.domain, EX.Dish))
g.add((EX.hasIngredient, RDFS.range, EX.Ingredient))

#subpropertyof 
g.add((EX.hasIngredient, RDFS.subPropertyOf, EX.usesIngredient))

#3
#print classes 
print("classes:")
for s in g.subjects(RDF.type, RDFS.Class):
    print(s)

#4
#print properties  
target_class = EX.Dish 

print(f"\nproperties of class {target_class}:")

domain_props = set(g.subjects(RDFS.domain, target_class))
if domain_props:
    for p in domain_props:
        print("  ", p)
else:
    print("   (none found via rdfs:domain)")

#5
#print all instances 
print("\ninstances:")
for s in g.subjects(RDF.type, None):
    print(s, " --> ", list(g.objects(s, RDF.type)))

classes:
http://example.org/restaurant#Restaurant
http://example.org/restaurant#Chef
http://example.org/restaurant#Dish
http://example.org/restaurant#Ingredient

properties of class http://example.org/restaurant#Dish:
   http://example.org/restaurant#hasIngredient

instances:
http://purl.org/heals/ingredient/AlmondMeal  -->  [rdflib.term.URIRef('http://purl.obolibrary.org/obo/FOODON_03400662'), rdflib.term.URIRef('http://purl.obolibrary.org/obo/FOODON_03400685'), rdflib.term.URIRef('http://purl.org/heals/food/Ingredient'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#NamedIndividual')]
http://purl.org/heals/ingredient/BrownSugar  -->  [rdflib.term.URIRef('http://purl.obolibrary.org/obo/FOODON_03400662'), rdflib.term.URIRef('http://purl.org/heals/food/Ingredient'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#NamedIndividual')]
http://purl.org/heals/ingredient/CaneSugar  -->  [rdflib.term.URIRef('http://purl.obolibrary.org/obo/FOODON_03400662'), rdflib.term.URIRef('http://purl

##### #6  
An RDF vocabulary is a structured set of URIs that define the terms used in a knowledge graph. These terms include classes (such as rdfs:Class), properties (rdf:Property), and semantic relationships such as rdfs:subClassOf, rdfs:domain, and rdfs:range. All vocabulary terms typically have a common namespace, which groups them together and makes them uniquely identifiable on the web. An RDF vocabulary defines meaning and structure, which allows data to be interpreted in a consistent manner across different systems.

## RDFS inferencing

The inference engine in owlrl is triggered by `DeductiveClosure`, which computes the closure of the graph. This requires us to specify under which semantic regime we want to perform the inference (e.g. what kind of rules under the RDFS, OWL, etc. semantics we want the reasoner to produce derivations on). For RDFS semantics we use `RDFS_Semantics` as parameter. See extra options [here](https://owl-rl.readthedocs.io/en/latest/stubs/owlrl.html#module-owlrl)


**Exercise 2**
1. expand the graph through RDFS semantics inference
2. print how many triples the new graph has
3. print out the triples in your new graph and inspect them. 

In [3]:
#TIP:always look at linked documentation 
#1
asserted_triple_count = len(g)
DeductiveClosure(RDFS_Semantics).expand(g) 

#2
print("asserted triples:", asserted_triple_count)
print("triples after RDFS inference:", len(g))
print("newly inferred triples (difference):", len(g) - asserted_triple_count)

#3
print("\ntriples in the inferred graph")
for s, p, o in g:
    print(s, p, o)

asserted triples: 852
triples after RDFS inference: 1493
newly inferred triples (difference): 641

triples in the inferred graph
a cut of beef and is part of the sub primal cut known as the chuck http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Resource
cheese http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Resource
http://purl.org/heals/ingredient/Garlic http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://purl.org/heals/food/Ingredient
http://purl.org/heals/ingredient/CanolaOil http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Resource
http://purl.obolibrary.org/obo/FOODON_03400673 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Resource
http://purl.org/heals/ingredient/WhiteVinegar http://www.w3.org/2004/02/skos/core#definition a sour liquid consisting of dilute and impure acetic acid
http://example.org/restaurant#makesDish http://www.w3.

## The explicit (asserted) graph vs the implicit (derived) graph, and RDF entailment

Asserted triples are those that are explicitly stated, while derived or inferred triples are those that are implicitly stated through the semantics of RDFS. 

**Exercise 3**

1. Write here code to generate a graph that contains **RDFS derived triples only** from yourRDF Knowledge Graph, not the asserted ones. See a clue on rdflib graph algebra [here](https://rdflib.readthedocs.io/en/stable/merging.html)
2. have a look at the inferred graph. Based on the RDFS semantics, explain for each triple the rule that was used to generate it.
3. Explain the concept RDF entailment, and the types of entailment RDFS can produce


In [4]:
#1
g_asserted = Graph()
g_asserted.parse("./data/yourRDF.ttl", format="turtle")

# copying asserted graph into new one
g_inferred = Graph()
g_inferred += g_asserted

#RDFS reasoning
DeductiveClosure(RDFS_Semantics).expand(g_inferred)

# subtracting triples
g_derived_only = g_inferred - g_asserted

print("asserted triples:", len(g_asserted))
print("after inference:", len(g_inferred))
print("derived-only triples:", len(g_derived_only))

print("\nderived triples only:")
for s, p, o in g_derived_only:
    print(s, p, o)

asserted triples: 5
after inference: 22
derived-only triples: 17

derived triples only:
http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://www.w3.org/2000/01/rdf-schema#subPropertyOf
http://www.w3.org/2000/01/rdf-schema#label http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://www.w3.org/2000/01/rdf-schema#label
http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
https://example.org/Mammalia http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Resource
https://example.org/whale http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Resource
https://example.org/Amphibia http://www.w3.org/1999/02/22-rdf-syntax-ns#type htt

#### #2
The graph was extended using the DeductiveClosure function with RDFS_Semantics. This means that RDFS inference rules were applied to the original (asserted) graph in order to infer new triples. After this step, the number of triples was increased, which indicates that new knowledge was inferred from the schema information already present in the graph.

The inferred triples were generated according to rules such as rdfs:subClassOf, rdfs:subPropertyOf, rdfs:domain, and rdfs:range. For instance, if a property has a defined domain, then the subject of the property is inferred to be an instance of the class. If a property has a defined range, then the object is inferred to be an instance of the class. In this manner, RDFS inference infers implicit knowledge by adding logically derived triples to the graph.

#### #3

RDF entailment is the concept that one graph implies another graph based on RDF or RDFS rules. When a graph entails a triple, it means that the triple is necessarily true whenever the original graph is true. Essentially, entailment is the concept of what can be logically deduced from the information already provided.

RDFS entailment is an extension of the above concept that takes advantage of schema information such as class hierarchies, property hierarchies, and domain and range information. Because of these rules, RDFS can automatically infer new type statements and new relationships among resources. This is because the hidden or implicit knowledge in the graph can be made explicit through reasoning, making the graph more informative without having to add new triples.

## Restaurant Assignment - Part 2


**Exercise 4**
1. load ingredients.rdf and recipes.rdf in one graph. The graph contains types of individuals and types of relationships between them. Print all the classes and properties in the combined graph with the namespace `ind` and the `wtm` namespace/vocabulary (`http://purl.org/heals/food/`). 

2. extend the `ind` vocabulary (`http://purl.org/heals/ingredient/`) by creating a hierarchy of ingredients (**hint: http://purl.org/heals/ingredient/CoconutMilk rdf:subClassOf http://purl.org/heals/ingredient/PlantMilk), and make these superclasses human readable by giving them labels**) 
3. do the same for the `wtm` vocabulary: add a hierarchy of recipes as well as a hierarchy of properties (**hint: http://purl.org/heals/food/hasCookingTemperature rdf:subPropertyOf ...) 
4. print the entailed triples as we did in the previous exercise
5. give three examples of how RDF semantics could aid the chefs in your restaurant 
    
6. which properties and classes could you add to the `wtm` and `ind` vocabularies to further describe your recipe and ingredient knowledge graph, aiding the chefs in your restaurant?  


In [5]:
#In order to extend the vocabularies you have to come up with multiple examples that follows the given hierarchy

#1)
g_food = Graph()

IND = Namespace("http://purl.org/heals/ingredient/")
WTM = Namespace("http://purl.org/heals/food/")

g_food.bind("ind", IND)
g_food.bind("wtm", WTM)
g_food.bind("rdf", RDF)
g_food.bind("rdfs", RDFS)
g_food.bind("owl", OWL)

g_food.parse("./data/ingredients.rdf", format="xml")
g_food.parse("./data/recipes.rdf", format="xml")

print("total triples (combined):", len(g_food))

#if a URI belongs to a namespace
def in_ns(uri, ns):
    return str(uri).startswith(str(ns))


print("\nclasses in IND or WTM")
classes = set()

# classes declared as rdfs:class or owl:class
for c in g.subjects(RDF.type, RDFS.Class):
    if in_ns(c, IND) or in_ns(c, WTM):
        classes.add(c)

for c in g.subjects(RDF.type, OWL.Class):
    if in_ns(c, IND) or in_ns(c, WTM):
        classes.add(c)

for c in sorted(classes, key=str):
    label = next(g.objects(c, RDFS.label), None)
    print("-", c, f'("{label}")' if label else "")

print("\nproperties in IND or WTM")
properties = set()

for p in g.subjects(RDF.type, RDF.Property):
    if in_ns(p, IND) or in_ns(p, WTM):
        properties.add(p)

for p in g.subjects(RDF.type, OWL.ObjectProperty):
    if in_ns(p, IND) or in_ns(p, WTM):
        properties.add(p)

for p in g.subjects(RDF.type, OWL.DatatypeProperty):
    if in_ns(p, IND) or in_ns(p, WTM):
        properties.add(p)

for p in sorted(properties, key=str):
    label = next(g.objects(p, RDFS.label), None)
    print("-", p, f'("{label}")' if label else "")


#2
#superclasses
g.add((IND.DairyAlternative, RDF.type, RDFS.Class))
g.add((IND.PlantMilk, RDF.type, RDFS.Class))
g.add((IND.Nut, RDF.type, RDFS.Class))
g.add((IND.NutFlour, RDF.type, RDFS.Class))
g.add((IND.Meat, RDF.type, RDFS.Class))
g.add((IND.Sweetener, RDF.type, RDFS.Class))
g.add((IND.Vinegar, RDF.type, RDFS.Class))

g.add((IND.DairyAlternative, RDFS.label, Literal("Dairy alternative")))
g.add((IND.PlantMilk, RDFS.label, Literal("Plant milk")))
g.add((IND.Nut, RDFS.label, Literal("Nut")))
g.add((IND.NutFlour, RDFS.label, Literal("Nut flour")))
g.add((IND.Meat, RDFS.label, Literal("Meat")))
g.add((IND.Sweetener, RDFS.label, Literal("Sweetener")))
g.add((IND.Vinegar, RDFS.label, Literal("Vinegar")))

#small class hierarchy
g.add((IND.PlantMilk, RDFS.subClassOf, IND.DairyAlternative))
g.add((IND.NutFlour, RDFS.subClassOf, IND.Nut))

#connect
for ing in [IND.CoconutMilk, IND.Almond, IND.Pecan, IND.Walnut, IND.AlmondMeal,
            IND.BrownSugar, IND.WhiteSugar, IND.CaneSugar, IND.WhiteVinegar, IND.AppleCiderVinegar,
            IND.Bacon, IND.Lamb]:
    g.add((ing, RDF.type, RDFS.Class))

#examples
g.add((IND.CoconutMilk, RDFS.subClassOf, IND.PlantMilk))

g.add((IND.Almond, RDFS.subClassOf, IND.Nut))
g.add((IND.Pecan, RDFS.subClassOf, IND.Nut))
g.add((IND.Walnut, RDFS.subClassOf, IND.Nut))

g.add((IND.AlmondMeal, RDFS.subClassOf, IND.NutFlour))

g.add((IND.BrownSugar, RDFS.subClassOf, IND.Sweetener))
g.add((IND.WhiteSugar, RDFS.subClassOf, IND.Sweetener))
g.add((IND.CaneSugar, RDFS.subClassOf, IND.Sweetener))

g.add((IND.WhiteVinegar, RDFS.subClassOf, IND.Vinegar))
g.add((IND.AppleCiderVinegar, RDFS.subClassOf, IND.Vinegar))

g.add((IND.Bacon, RDFS.subClassOf, IND.Meat))
g.add((IND.Lamb, RDFS.subClassOf, IND.Meat))


#3
#recipe hierarchy- classes
g.add((WTM.Recipe, RDF.type, RDFS.Class))
g.add((WTM.DessertRecipe, RDF.type, RDFS.Class))
g.add((WTM.MainCourseRecipe, RDF.type, RDFS.Class))
g.add((WTM.SauceRecipe, RDF.type, RDFS.Class))

g.add((WTM.Recipe, RDFS.label, Literal("Recipe")))
g.add((WTM.DessertRecipe, RDFS.label, Literal("Dessert recipe")))
g.add((WTM.MainCourseRecipe, RDFS.label, Literal("Main course recipe")))
g.add((WTM.SauceRecipe, RDFS.label, Literal("Sauce recipe")))

g.add((WTM.DessertRecipe, RDFS.subClassOf, WTM.Recipe))
g.add((WTM.MainCourseRecipe, RDFS.subClassOf, WTM.Recipe))
g.add((WTM.SauceRecipe, RDFS.subClassOf, WTM.Recipe))

#property hierarchy- properties
g.add((WTM.hasCookingParameter, RDF.type, RDF.Property))
g.add((WTM.hasCookingTemperature, RDF.type, RDF.Property))
g.add((WTM.hasCookingTime, RDF.type, RDF.Property))
g.add((WTM.hasBakingTime, RDF.type, RDF.Property))

g.add((WTM.hasCookingParameter, RDFS.label, Literal("has cooking parameter")))
g.add((WTM.hasCookingTemperature, RDFS.label, Literal("has cooking temperature")))
g.add((WTM.hasCookingTime, RDFS.label, Literal("has cooking time")))
g.add((WTM.hasBakingTime, RDFS.label, Literal("has baking time")))

#examples - subPropertyOf
g.add((WTM.hasCookingTemperature, RDFS.subPropertyOf, WTM.hasCookingParameter))
g.add((WTM.hasBakingTime, RDFS.subPropertyOf, WTM.hasCookingTime))


#4
g_asserted = Graph()
g_asserted += g 

g_inferred = Graph()
g_inferred += g_asserted
DeductiveClosure(RDFS_Semantics).expand(g_inferred)

g_derived_only = g_inferred - g_asserted

print("\nasserted triples - after extesiuob:", len(g_asserted))
print("triples after RDFS inference:", len(g_inferred))
print("derived triples:", len(g_derived_only))

print("\nderived triples (first 200)")
for i, (s, p, o) in enumerate(g_derived_only):
    if i >= 200:
        break
    print(s, p, o)

total triples (combined): 1299

classes in IND or WTM

properties in IND or WTM
- http://purl.org/heals/food/dislikes 
- http://purl.org/heals/food/forbids 
- http://purl.org/heals/food/hasGluten 
- http://purl.org/heals/food/hasGlycemicIndex 
- http://purl.org/heals/food/hasTexture 
- http://purl.org/heals/food/isAllergicTo 
- http://purl.org/heals/food/substitutesFor 

asserted triples - after extesiuob: 1554
triples after RDFS inference: 1667
derived triples: 113

derived triples (first 200)
http://purl.org/heals/ingredient/AppleCiderVinegar http://www.w3.org/2000/01/rdf-schema#subClassOf http://purl.org/heals/ingredient/AppleCiderVinegar
http://purl.org/heals/ingredient/NutFlour http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Resource
http://purl.org/heals/ingredient/Pecan http://www.w3.org/2000/01/rdf-schema#subClassOf http://www.w3.org/2000/01/rdf-schema#Resource
http://www.w3.org/2000/01/rdf-schema#subPropertyOf http://www.w3.org/1999/02/22-r

#### #5
1. Safer filtering for dietary needs: If “CoconutMilk” is a subclass of “PlantMilk” and “PlantMilk” is a “DairyAlternative,” then chefs can search for “DairyAlternative” and automatically find coconut milk without listing every plant milk manually.

2.	Automatic recipe options through hierarchy: If a recipe is tagged as a “DessertRecipe,” it also becomes a “Recipe” through rdfs:subClassOf, so a general “show me all recipes” query will still include all desserts automatically.

3.	More flexible querying via property hierarchies: If hasCookingTemperature is a sub-property of hasCookingParameter, then any query for “cooking parameters” will also include temperatures, without writing separate queries for each specific property.

#### #6

Add to ind:

-> Classes: Allergen (with subclasses like TreeNutAllergen, GlutenAllergen), Dairy, Spice, Vegetable, Seafood, Legume

-> Properties: hasAllergen, isVegan, isHalal, isKosher, hasNutritionValue, hasSubstitute


Add to wtm:

-> Classes: VeganRecipe, GlutenFreeRecipe, QuickRecipe, HighProteinRecipe

-> Properties: hasDifficulty, hasPrepTime, hasCookTime, serves, requiresEquipment, hasCuisine, hasDietTag

## HI ontology exploration

In your project, you will be working with a Hybrid Intelligence (HI) ontology. After acquanting yourself with its structure in the previous tutorial, you will now expand it yourself. Using the tools from the exercises above, perform the following actions:

1. Load the HI ontology from the data folder (hi_ontology.ttl) with RDFlib (don't forget to define a Namespace).
2. Create 3 new subclasses using the rdfs:subClassOf predicate.
3. Create at least one domain and one range restriction using rdfs.
4. Populate each class you've created with at least one new instance each (hint: use rdf:type).



In [6]:

#1
g_hi = Graph()

HI = Namespace("http://example.org/hi#") 
g_hi.bind("hi", HI)
g_hi.bind("rdf", RDF)
g_hi.bind("rdfs", RDFS)

g_hi.parse("./data/hi_ontology.ttl", format="turtle")

print("Triples loaded:", len(g_hi))


#2
g_hi.add((HI.HumanAgent, RDF.type, RDFS.Class))
g_hi.add((HI.AI_Agent, RDF.type, RDFS.Class))
g_hi.add((HI.HybridTeam, RDF.type, RDFS.Class))

# subclass relations
g_hi.add((HI.HumanAgent, RDFS.subClassOf, HI.Agent))
g_hi.add((HI.AI_Agent, RDFS.subClassOf, HI.Agent))
g_hi.add((HI.HybridTeam, RDFS.subClassOf, HI.CollaborativeSystem))

# optional labels
g_hi.add((HI.HumanAgent, RDFS.label, Literal("Human Agent")))
g_hi.add((HI.AI_Agent, RDFS.label, Literal("AI Agent")))
g_hi.add((HI.HybridTeam, RDFS.label, Literal("Hybrid Team")))


#3

g_hi.add((HI.collaboratesWith, RDF.type, RDF.Property))

# domain: only agents can collaborate
g_hi.add((HI.collaboratesWith, RDFS.domain, HI.Agent))

# range: collaboration is with another agent
g_hi.add((HI.collaboratesWith, RDFS.range, HI.Agent))

g_hi.add((HI.collaboratesWith, RDFS.label, Literal("collaborates with")))


#4

g_hi.add((HI.Alice, RDF.type, HI.HumanAgent))
g_hi.add((HI.GPT_System, RDF.type, HI.AI_Agent))
g_hi.add((HI.TeamAlpha, RDF.type, HI.HybridTeam))

# example on collaboration
g_hi.add((HI.Alice, HI.collaboratesWith, HI.GPT_System))


print("triples after extension:", len(g_hi))

print("\nnew triples added")
for s, p, o in g_hi:
    print(s, p, o)

Triples loaded: 114
triples after extension: 131

new triples added
http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Learning http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#NamedIndividual
http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Adult http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Human
http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Interaction http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#Class
http://example.org/hi#collaboratesWith http://www.w3.org/2000/01/rdf-schema#range http://example.org/hi#Agent
http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/interactionTask http://www.w3.org/2000/01/rdf-schema#domain http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/Interaction
http://www.semanticweb.org/vbr240/ontologies/2022/4/u